# Static Accuracy Analysis (thesis-grade)

Loads the most recent `static_accuracy` run and reports static accuracy in the Y-Z plane (Volcaniarm is 2-DOF planar; X is fixed bias).

**Headline metric**: `d_error = d_detected - d_urdf`, signed, computed in the **world frame**:
- `d_detected = ||(apriltag_marker_ee.origin - apriltag_marker_base.origin)_yz||` in `world`
- `d_urdf     = ||(apriltag_ee_link.origin   - apriltag_base_link.origin)_yz||`   in `world`

Both origins are looked up in a single common world-aligned frame before the difference is taken, so Y and Z refer to the same physical axes for detection and URDF (the apriltag marker frames have different parent orientations than the URDF tag links — comparing their parent-frame components directly is wrong).

The runner logs `d_detected`, `d_urdf`, `d_error` directly into `tag_observations.csv` at capture time. **Application zones**: `acceptable <= 10 mm`, `marginal 10-30 mm`, `failing > 30 mm` (tunable in `analysis/metrics.py`). Repeatability scatter lives in the separate `repeatability.ipynb` notebook.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from volcaniarm_calibration.analysis import (
    accuracy_segment_length, align_fk_to_tag, latest_run, list_runs, load_run,
    repeatability_iso9283, summary, threshold_color, threshold_zone,
    WEEDING_ACCEPTABLE_MM, WEEDING_MARGINAL_MM,
)

TEST_NAME = 'static_accuracy'
RUN_DIR = latest_run(TEST_NAME)
run = load_run(RUN_DIR)
config, tag, fk = run['config'], run['tag'], run['fk']
print(f'run_id:       {config.get("run_id")}')
print(f'status:       {config.get("status")}  failure_reason: {config.get("failure_reason")}')
print(f'goal (y, z):  {config["goals"][0]}')
print(f'iterations:   {config["num_cycles"]}')
print(f'tag rows:     {len(tag)}')
print(f'fk rows:      {len(fk)}')

target = tag[tag['phase'] == 'target'].copy()
home = tag[tag['phase'] == 'home'].copy()

## Accuracy: detected vs URDF Y-Z origin distance (world frame)

For each target visit the runner captures both:
- `d_detected = ||(apriltag_marker_ee.origin - apriltag_marker_base.origin)_yz||` in `world`
- `d_urdf     = ||(apriltag_ee_link.origin   - apriltag_base_link.origin)_yz||`   in `world`

Their difference, `d_error = d_detected - d_urdf`, is the static accuracy residual: how far the detected EE marker's distance-from-base disagrees with the URDF prediction at the same instant. Because both origins are looked up in the same world-aligned frame, the Y and Z components refer to the same physical axes for detection and URDF.

In [ ]:
acc = accuracy_segment_length(tag)
print(f'd_error across {acc.n} target visits:')
print(f'  {acc.in_mm()}')
zone = threshold_zone(abs(acc.mean) * 1000)
print(f'  mean |d_error| zone: {zone}')

per_visit = target[['cycle', 'target_idx', 'd_detected', 'd_urdf', 'd_error']].copy()
per_visit[['d_detected_mm', 'd_urdf_mm', 'd_error_mm']] = (
    per_visit[['d_detected', 'd_urdf', 'd_error']] * 1000
)
print('\nper-visit:')
print(per_visit[['cycle', 'target_idx', 'd_detected_mm', 'd_urdf_mm', 'd_error_mm']]
      .to_string(index=False, float_format=lambda v: f'{v:8.3f}'))

## Distribution of `d_error` across visits

Histogram of the per-visit accuracy residual, with the application threshold lines for context. Centred close to zero = the URDF mount values agree with the detected geometry. A consistent offset = bias in the URDF tag mounts that should be calibrated into the xacro.

In [ ]:
d_err_mm = (target['d_error'].dropna().to_numpy() * 1000)
if d_err_mm.size == 0:
    print('no d_error rows in this run.')
else:
    fig, ax = plt.subplots(1, 1, figsize=(8, 4))
    ax.hist(d_err_mm, bins=max(5, len(d_err_mm) // 2),
            color='steelblue', edgecolor='white', alpha=0.85)
    ax.axvline(0.0, color='black', linestyle='-', linewidth=1, alpha=0.5)
    ax.axvline(d_err_mm.mean(), color='#2e9c4a', linestyle='--', linewidth=2,
               label=f'mean = {d_err_mm.mean():+.2f} mm')
    for lim_mm, style, lbl in (
        (WEEDING_ACCEPTABLE_MM, ':', f'+/- acceptable ({WEEDING_ACCEPTABLE_MM:.0f} mm)'),
        (WEEDING_MARGINAL_MM, '--', f'+/- marginal ({WEEDING_MARGINAL_MM:.0f} mm)'),
    ):
        ax.axvline(+lim_mm, color='gray', linestyle=style, alpha=0.6, label=lbl)
        ax.axvline(-lim_mm, color='gray', linestyle=style, alpha=0.6)
    ax.set_xlabel('d_error [mm]  (detected - urdf, signed)')
    ax.set_ylabel('count')
    ax.set_title(f'd_error distribution  ({config["run_id"]})')
    ax.grid(True, alpha=0.3)
    ax.legend(fontsize=8, loc='best')
    fig.tight_layout()

## Notes

- `d_error` is the only metric this notebook reports. It's frame-independent: scalar Y-Z segment-length comparison cancels any apriltag-vs-URDF axis convention difference.
- Application zones (`acceptable`, `marginal`, `failing`) live in `analysis/metrics.py` and currently target weeding (1 cm typical, 3 cm marginal). Re-tune for the actuator head you use.
- For thesis-quality stats, run with `iterations >= 10` so the histogram is meaningful and the 95% CI tightens. With one sample per visit, total sample count equals iteration count.